# Publication-safe notebook

This generalized reference implementation expects a live SparkSession named
`spark` and pseudonymous tables matching `docs/data-contract.md`. It contains no
institution-specific ingestion, credentials, internal paths, data, or results.

Parameter defaults are illustrative and intentionally differ from internal
operating settings. Calibrate them only with authorized development data.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import numpy as np
import pandas as pd

import os

WRITE_DB = os.environ.get("FRAUD_GRAPH_WRITE_DB", "fraud_graph_demo")
EMP = os.environ.get("FRAUD_GRAPH_PREFIX", "onboarding_fraud")
IN_FEATURES = f"{WRITE_DB}.{EMP}_v2_final_features"

FEATURES = ["n_shared_mule", "closeness_to_bad", "out_deg", "n_shared_atm",
            "comm_bad_rate", "in_dense_block", "ppr"]

TRAIN_ANCHORS = ["202401", "202402"]
K_PCTS = [float(value) for value in os.environ.get("FRAUD_GRAPH_EVALUATION_PCTS", "0.02,0.05").split(",")]
MAX_BINS = int(os.environ.get("FRAUD_GRAPH_MAX_BINS", "10"))
L2_LAMBDA = float(os.environ.get("FRAUD_GRAPH_L2_LAMBDA", "1.0"))
IRLS_ITERS = int(os.environ.get("FRAUD_GRAPH_IRLS_ITERS", "25"))
RULE_FEATURES = ["in_dense_block", "n_shared_atm", "n_shared_mule"]

OUT_WOE    = f"{WRITE_DB}.{EMP}_v2_scorecard_woe"
OUT_COEF   = f"{WRITE_DB}.{EMP}_v2_scorecard_coef"
OUT_SCORES = f"{WRITE_DB}.{EMP}_v2_scorecard_scores"
OUT_VALID  = f"{WRITE_DB}.{EMP}_v2_scorecard_validation"

spark.conf.set("spark.sql.shuffle.partitions", os.environ.get("FRAUD_GRAPH_SHUFFLE_PARTITIONS", "200"))

def save_overwrite(df, t):
    spark.sql("DROP TABLE IF EXISTS " + t); df.write.mode("overwrite").saveAsTable(t)

print("scorecard | features:", FEATURES, "| train:", TRAIN_ANCHORS, "| L2:", L2_LAMBDA, "| bins:", MAX_BINS)


In [ ]:
pdf = spark.table(IN_FEATURES).select("cust", "anchor_month", "target_fraud", *FEATURES).toPandas()
pdf["target_fraud"] = pdf["target_fraud"].fillna(0).astype(int)
for c in FEATURES:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(0.0)
ANCHORS = sorted(pdf["anchor_month"].unique().tolist())
TRAIN = [a for a in TRAIN_ANCHORS if a in ANCHORS] or ANCHORS
tr_mask = pdf["anchor_month"].isin(TRAIN).values
print("frame:", pdf.shape, "| anchors:", ANCHORS, "| train:", TRAIN,
      "| bad by anchor:", {a: int(pdf.loc[pdf.anchor_month == a, "target_fraud"].sum()) for a in ANCHORS})

def capture_at_pd(y, score, ks):
    y = np.asarray(y, dtype=int); score = np.asarray(score, dtype=float)
    order = np.argsort(-score, kind="mergesort"); ys = y[order]; n = len(y); nbad = int(y.sum()); rows = []
    for p in ks:
        K = max(1, int(n * p)); topbad = int(ys[:K].sum())
        cap = (topbad / nbad) if nbad else 0.0
        lift = ((topbad / K) / (nbad / n)) if nbad else 0.0
        rows.append((p, K, topbad, cap, lift))
    return rows, n, nbad

def _pctrank_group(s):
    n = len(s)
    if n <= 1: return pd.Series(np.zeros(n), index=s.index)
    return (s.rank(method="min") - 1.0) / (n - 1.0)


In [ ]:
def fit_bins(x, max_bins=MAX_BINS):
    s = pd.Series(np.asarray(x, dtype=float)); nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1:
        return {"kind": "const"}
    if nuniq <= max_bins:
        return {"kind": "cat", "vals": sorted(s.dropna().unique().tolist())}
    nz = s[s != 0]
    if (s == 0).mean() >= 0.02 and len(nz) >= max_bins:
        edges = np.unique(np.quantile(nz, np.linspace(0, 1, max_bins)))
        return {"kind": "zeroq", "edges": edges}
    edges = np.unique(np.quantile(s, np.linspace(0, 1, max_bins + 1)))
    return {"kind": "q", "edges": edges}

def apply_bins(x, spec):
    x = np.asarray(x, dtype=float)
    if spec["kind"] == "const":
        return np.zeros(len(x), dtype=int)
    if spec["kind"] == "cat":
        m = {v: i for i, v in enumerate(spec["vals"])}
        return np.array([m.get(v, 0) for v in x], dtype=int)
    if spec["kind"] == "zeroq":
        out = np.zeros(len(x), dtype=int); nz = x != 0
        inner = spec["edges"][1:-1] if len(spec["edges"]) > 2 else spec["edges"][1:]
        out[nz] = 1 + np.digitize(x[nz], inner)
        return out
    inner = spec["edges"][1:-1] if len(spec["edges"]) > 2 else spec["edges"][1:]
    return np.digitize(x, inner)

def woe_map(bins_tr, y_tr):
    d = pd.DataFrame({"b": bins_tr, "y": np.asarray(y_tr, dtype=int)})
    g = d.groupby("b").agg(n=("y", "size"), bad=("y", "sum"))
    tot_bad = max(int(g["bad"].sum()), 1); tot_good = max(int((g["n"] - g["bad"]).sum()), 1)
    pg = ((g["n"] - g["bad"]) / tot_good).clip(lower=1e-6)
    pb = (g["bad"] / tot_bad).clip(lower=1e-6)
    g["woe"] = np.log(pb / pg)
    iv = float(((pb - pg) * g["woe"]).sum())
    return g["woe"].to_dict(), iv, g

y_all = pdf["target_fraud"].values
y_tr  = pdf.loc[tr_mask, "target_fraud"].values
SPECS = {}; WOE_MAPS = {}; IVS = {}; woe_rows = []
WOE = np.zeros((len(pdf), len(FEATURES)), dtype=float)
print("=== per-feature WOE / IV (fit on train) ===")
for j, f in enumerate(FEATURES):
    spec = fit_bins(pdf.loc[tr_mask, f].values)
    b_all = apply_bins(pdf[f].values, spec)
    wmap, iv, g = woe_map(b_all[tr_mask], y_tr)
    SPECS[f] = spec; WOE_MAPS[f] = wmap; IVS[f] = iv
    WOE[:, j] = np.array([wmap.get(int(b), 0.0) for b in b_all])
    print("   %-20s IV=%.3f  bins=%d" % (f, iv, len(g)))
    for b, w in wmap.items():
        woe_rows.append((f, int(b), float(w), round(iv, 4)))
save_overwrite(spark.createDataFrame(woe_rows, ["feature", "bin", "woe", "iv"]), OUT_WOE)
print("saved WOE table:", OUT_WOE)


In [ ]:
mu = WOE[tr_mask].mean(axis=0); sd = WOE[tr_mask].std(axis=0); sd[sd < 1e-9] = 1.0
Xz_all = (WOE - mu) / sd
Xtr = np.column_stack([np.ones(int(tr_mask.sum())), Xz_all[tr_mask]])
d = Xtr.shape[1]; w = np.zeros(d)
pen = np.full(d, L2_LAMBDA); pen[0] = 0.0
for _ in range(IRLS_ITERS):
    p = 1.0 / (1.0 + np.exp(-np.clip(Xtr @ w, -30, 30)))
    Wd = np.clip(p * (1.0 - p), 1e-6, None)
    grad = Xtr.T @ (y_tr - p) - pen * w
    H = Xtr.T @ (Xtr * Wd[:, None]) + np.diag(pen) + 1e-6 * np.eye(d)
    w = w + np.linalg.solve(H, grad)
intercept = float(w[0]); coefs = w[1:]
print("=== fitted scorecard (train %s) ===" % "+".join(TRAIN))
print("   intercept = %.4f" % intercept)
coef_rows = []
for f, c in sorted(zip(FEATURES, coefs), key=lambda t: -abs(t[1])):
    print("   %-20s coef=%+.4f   IV=%.3f" % (f, c, IVS[f]))
    coef_rows.append((f, float(c), round(IVS[f], 4)))
save_overwrite(spark.createDataFrame(
    [("__intercept__", intercept, 0.0)] + coef_rows, ["feature", "coef", "iv"]), OUT_COEF)
print("saved coefficients:", OUT_COEF)


In [ ]:
logit = intercept + Xz_all @ coefs
pdf["model_score"] = 1.0 / (1.0 + np.exp(-np.clip(logit, -30, 30)))
pdf["risk_pctl"] = pdf.groupby("anchor_month")["model_score"].rank(method="average", pct=True).values * 100.0
scores_pd = pdf[["cust", "anchor_month", "target_fraud", "model_score", "risk_pctl"]].copy()
scores_pd["target_fraud"] = scores_pd["target_fraud"].astype(int)
scores_pd["model_score"] = scores_pd["model_score"].astype(float)
scores_pd["risk_pctl"] = scores_pd["risk_pctl"].astype(float)
save_overwrite(spark.createDataFrame(scores_pd), OUT_SCORES)
print("saved model scores:", OUT_SCORES, "| rows:", len(scores_pd))


In [ ]:
PR = {c: pdf.groupby("anchor_month")[c].transform(_pctrank_group).fillna(0.0).values for c in FEATURES}
def blend(cols):
    return np.sum([PR[c] for c in cols], axis=0)
SCORES = {"MODEL": pdf["model_score"].values,
          "BLEND06_equalwt": blend(["n_shared_mule", "closeness_to_bad", "out_deg", "n_shared_atm"]),
          "04core_baseline": blend(["closeness_to_bad", "n_shared_mule"])}

valid_rows = []
print("=== capture@K : MODEL vs BLEND06 vs 04-core ===")
for name, sc in SCORES.items():
    agg = {p: [] for p in K_PCTS}; aggl = {p: [] for p in K_PCTS}
    for a in ANCHORS:
        m = (pdf.anchor_month == a).values
        rows, n, nbad = capture_at_pd(y_all[m], sc[m], K_PCTS)
        for (p, K, tb, cap, lift) in rows:
            valid_rows.append((name, a, p, K, n, nbad, tb, round(cap, 4), round(lift, 3)))
            agg[p].append(cap); aggl[p].append(lift)
    for p in K_PCTS:
        mc, ml = float(np.mean(agg[p])), float(np.mean(aggl[p]))
        valid_rows.append((name, "MEAN", p, -1, -1, -1, -1, round(mc, 4), round(ml, 3)))
        print("   %-16s MEAN top%4.0f%%  capture=%.4f  lift=%.2f" % (name, p * 100, mc, ml))
save_overwrite(spark.createDataFrame(valid_rows, ["model", "scope", "top_pct", "K", "n_cust", "n_bad",
                                                  "bad_in_topK", "capture", "lift"]), OUT_VALID)
print("saved validation:", OUT_VALID)


In [ ]:
def fit_score(train_anchors):
    trm = pdf["anchor_month"].isin(train_anchors).values
    ytr = pdf.loc[trm, "target_fraud"].values
    Wm = np.zeros((len(pdf), len(FEATURES)))
    for j, f in enumerate(FEATURES):
        sp = fit_bins(pdf.loc[trm, f].values); ba = apply_bins(pdf[f].values, sp)
        wm, _, _ = woe_map(ba[trm], ytr)
        Wm[:, j] = np.array([wm.get(int(b), 0.0) for b in ba])
    m_ = Wm[trm].mean(axis=0); s_ = Wm[trm].std(axis=0); s_[s_ < 1e-9] = 1.0
    Xz = (Wm - m_) / s_
    Xt = np.column_stack([np.ones(int(trm.sum())), Xz[trm]]); dd = Xt.shape[1]; ww = np.zeros(dd)
    pn = np.full(dd, L2_LAMBDA); pn[0] = 0.0
    for _ in range(IRLS_ITERS):
        pp = 1.0 / (1.0 + np.exp(-np.clip(Xt @ ww, -30, 30))); Wq = np.clip(pp * (1 - pp), 1e-6, None)
        gg = Xt.T @ (ytr - pp) - pn * ww
        HH = Xt.T @ (Xt * Wq[:, None]) + np.diag(pn) + 1e-6 * np.eye(dd)
        ww = ww + np.linalg.solve(HH, gg)
    return 1.0 / (1.0 + np.exp(-np.clip(ww[0] + Xz @ ww[1:], -30, 30)))

if len(ANCHORS) >= 2:
    print("=== model LOAO (capture@5% / @10% on the held-out anchor) ===")
    for held in ANCHORS:
        train = [a for a in ANCHORS if a != held]
        sc = fit_score(train); m = (pdf.anchor_month == held).values
        rows, _, _ = capture_at_pd(y_all[m], sc[m], [0.05, 0.10])
        print("   held-out %s: trained-on-%s -> capture@5%%=%.4f | @10%%=%.4f"
              % (held, "+".join(train), rows[0][3], rows[1][3]))
else:
    print("   (need >=2 anchors for LOAO)")


In [ ]:
def _woe_iv(cnt, bad):
    good = cnt - bad; tb, tg = bad.sum(), good.sum()
    pb = (bad / tb).clip(lower=1e-6) if tb else bad * 0 + 1e-6
    pg = (good / tg).clip(lower=1e-6) if tg else good * 0 + 1e-6
    return float(((pg - pb) * np.log(pg / pb)).sum())

def _robust_bins(x, max_bins=10):
    s = pd.Series(np.asarray(x, dtype=float)); nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1: return np.zeros(len(s), dtype=int)
    if nuniq <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0); out = np.zeros(len(s), dtype=int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = pd.qcut(s[pos].rank(method="first"), q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values; return out
    return pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values

for a in ANCHORS:
    dsl = pdf[pdf.anchor_month == a]; y = dsl["target_fraud"].astype(int).values; tot_bad = int(y.sum())
    b = _robust_bins(dsl["model_score"].astype(float).values, max_bins=10)
    g = pd.DataFrame({"b": b, "y": y}).groupby("b").agg(cnt=("y", "size"), bad=("y", "sum")).sort_index()
    er = (100.0 * g["bad"] / g["cnt"]).values; iv = _woe_iv(g["cnt"], g["bad"])
    mono = bool(np.all(np.diff(er) >= -1e-9) or np.all(np.diff(er) <= 1e-9))
    print("=== model_score sloping @ %s (bad=%d/n=%d)  IV=%.3f  bins=%d  mono=%s  top-bin=%.2f%% ==="
          % (a, tot_bad, len(dsl), iv, len(g), mono, er[-1] if len(er) else 0))

print("=== rule overlays (pooled) ===  base rate: %d / %d = %.3f%%"
      % (int(pdf.target_fraud.sum()), len(pdf), 100.0 * pdf.target_fraud.sum() / max(len(pdf), 1)))
tot_bad_all = int(pdf.target_fraud.sum()); n_all = len(pdf)
for rf in RULE_FEATURES:
    if rf not in pdf.columns: continue
    flag = pdf[rf].astype(float) > 0; nf = int(flag.sum()); bf = int(pdf.loc[flag, "target_fraud"].sum())
    prec = (bf / nf) if nf else 0.0
    print("   %-16s > 0 : flags %6d | %4d fraud | precision=%.2f%% | recall=%.2f%% | lift=%.1fx"
          % (rf, nf, bf, 100.0 * prec, 100.0 * (bf / tot_bad_all if tot_bad_all else 0),
             prec / (tot_bad_all / max(n_all, 1)) if tot_bad_all else 0.0))


In [ ]:
print("=== Notebook 07 (v2) summary ===")
print("WOE bins   :", OUT_WOE, "| coefficients:", OUT_COEF)
print("model scores:", OUT_SCORES, "| rows:", spark.table(OUT_SCORES).count())
print("validation :", OUT_VALID, "(MODEL vs BLEND06 vs 04-core, capture@K).")
print("Compare mean capture@5%/10% for MODEL and BLEND06. If they tie,")
print("the equal-weight blend was already near-optimal and the next gain must come from NEW features (nb 08).")
print("Example triage = rank by risk_pctl and review configured rule flags under human oversight.")
print("NOTEBOOK 07 (v2) COMPLETE — supervised WOE-logistic scorecard over the leak-free features.")


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import IPython.display as _disp
from pyspark.sql import functions as _F

CUST_TBL      = f"{WRITE_DB}.{EMP}_v2_scorecard_scores"
PLOT_FEATURES = ["model_score"]
PLOT_ANCHOR   = "202401"
MIN_BIN_FRAC = float(os.environ.get("FRAUD_GRAPH_MIN_BIN_FRACTION", "0.02"))
MIN_STABLE_CNT = int(os.environ.get("FRAUD_GRAPH_MIN_STABLE_COUNT", "30"))
MIN_STABLE_BAD = int(os.environ.get("FRAUD_GRAPH_MIN_STABLE_POSITIVES", "3"))
RISKY_ON_LEFT = True

def _wilson_lcb(bad, cnt, z=1.96):
    bad = np.asarray(bad, float); cnt = np.asarray(cnt, float)
    p = np.where(cnt > 0, bad / cnt, 0.0); denom = 1 + z*z/np.maximum(cnt, 1)
    centre = p + z*z/(2*np.maximum(cnt, 1)); margin = z*np.sqrt((p*(1-p) + z*z/(4*np.maximum(cnt,1)))/np.maximum(cnt,1))
    return np.where(cnt > 0, np.clip((centre - margin)/denom, 0, 1), 0.0)

def _robust_bins(x, max_bins=10):
    s = pd.Series(np.asarray(x, dtype=float)); nuniq = int(s.nunique(dropna=True))
    if nuniq <= 1: return np.zeros(len(s), dtype=int)
    if nuniq <= max_bins:
        order = {v: i for i, v in enumerate(sorted(s.dropna().unique()))}
        return s.map(order).fillna(0).astype(int).values
    z = (s == 0); out = np.zeros(len(s), dtype=int)
    if z.mean() >= 0.02 and int((~z).sum()) >= max_bins:
        pos = ~z
        qr = pd.qcut(s[pos].rank(method="first"), q=min(max_bins - 1, int((~z).sum())), labels=False, duplicates="drop")
        out[pos.values] = 1 + qr.astype(int).values; return out
    return pd.qcut(s.rank(method="first"), q=max_bins, labels=False, duplicates="drop").astype(int).values

def _club_sparse(d, n, min_frac):
    d = d.sort_index().copy()
    while len(d) > 2:
        share = d["cnt"] / n
        if share.min() >= min_frac: break
        j = int(share.values.argmin()); k = j + 1 if j == 0 else j - 1
        lo, hi = min(j, k), max(j, k)
        d.loc[d.index[lo], ["cnt", "bad"]] += d.loc[d.index[hi], ["cnt", "bad"]].values
        d.loc[d.index[lo], "hi"] = max(d.loc[d.index[lo], "hi"], d.loc[d.index[hi], "hi"])
        d.loc[d.index[lo], "lo"] = min(d.loc[d.index[lo], "lo"], d.loc[d.index[hi], "lo"])
        d = d.drop(d.index[hi]).reset_index(drop=True)
    return d

def _woe_iv(cnt, bad):
    good = cnt - bad; tb, tg = bad.sum(), good.sum()
    pb = (bad / tb).clip(lower=1e-6); pg = (good / tg).clip(lower=1e-6)
    return float(((pg - pb) * np.log(pg / pb)).sum())

_pdf = spark.table(CUST_TBL).where(_F.col("anchor_month") == PLOT_ANCHOR).select("target_fraud", *PLOT_FEATURES).toPandas()
_y = _pdf["target_fraud"].astype(int).values; _tot_bad = int(_y.sum()); _tot_good = len(_y) - _tot_bad
print("plotting anchor %s | customers=%d | frauds=%d" % (PLOT_ANCHOR, len(_pdf), _tot_bad))

def plot_sloping(feature):
    x = _pdf[feature].astype(float).values
    d = (pd.DataFrame({"b": _robust_bins(x), "x": x, "y": _y}).groupby("b")
         .agg(cnt=("y", "size"), bad=("y", "sum"), lo=("x", "min"), hi=("x", "max")).sort_index())
    d = _club_sparse(d, len(_pdf), MIN_BIN_FRAC)
    d = (d.iloc[::-1] if RISKY_ON_LEFT else d).reset_index(drop=True)
    d["event_rate_pct"] = 100.0 * d["bad"] / d["cnt"]
    cum_bad = d["bad"].cumsum(); cum_good = (d["cnt"] - d["bad"]).cumsum()
    d["pct_captured"] = 100.0 * cum_bad / max(_tot_bad, 1)
    ks = float((100.0 * cum_bad / max(_tot_bad, 1) - 100.0 * cum_good / max(_tot_good, 1)).abs().max())
    iv = _woe_iv(d["cnt"], d["bad"])
    d["er_lcb_pct"] = 100.0 * _wilson_lcb(d["bad"].values, d["cnt"].values)
    d["stable"] = (d["cnt"] >= MIN_STABLE_CNT) & (d["bad"] >= MIN_STABLE_BAD)
    labels = [("%.3g" % lo) if lo == hi else ("[%.3g, %.3g]" % (lo, hi)) for lo, hi in zip(d["lo"], d["hi"])]
    xs = range(len(d))
    fig, ax1 = plt.subplots(figsize=(11, 5))
    ax1.bar(xs, d["event_rate_pct"], color="#c97b84", alpha=0.85, label="Event Rate (%)")
    for i, v in enumerate(d["event_rate_pct"]): ax1.text(i, v, "%.2f" % v, ha="center", va="bottom", fontsize=8)
    ax1.set_ylabel("Event Rate (%)"); ax1.set_xticks(list(xs)); ax1.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    for i, st in enumerate(d["stable"].values):
        if not st: ax1.patches[i].set_hatch("///"); ax1.patches[i].set_edgecolor("#7a2a34")
    ax2 = ax1.twinx()
    ax2.plot(xs, d["pct_captured"], color="#a23b4e", marker="o", label="Cumulative % Fraud Captured")
    ax2.set_ylabel("Cumulative % Fraud Captured"); ax2.set_ylim(0, 105)
    ax1.set_title("%s @ %s   (IV=%.3f, KS=%.1f)   hatched = UNSTABLE (n<%d)\nEvent Rate (bars) & Cumulative %% Fraud Captured (line)"
                  % (feature, PLOT_ANCHOR, iv, ks, MIN_STABLE_CNT), fontsize=12)
    ax1.legend(loc="upper left"); ax2.legend(loc="lower right"); fig.tight_layout(); plt.show()
    _disp.display(pd.DataFrame({"bin": labels, "# frauds": d["bad"].astype(int).values,
                                "# customers": d["cnt"].astype(int).values,
                                "event_rate_%": d["event_rate_pct"].round(3).values,
                                "event_rate_LCB_%": d["er_lcb_pct"].round(2).values,
                                "cum_%_captured": d["pct_captured"].round(2).values,
                                "stable": d["stable"].values}))

for _f in PLOT_FEATURES:
    try: plot_sloping(_f)
    except Exception as _e: print("  %s: skipped (%s)" % (_f, _e))
